In [3]:
from sklearn.datasets import fetch_20newsgroups

data = fetch_20newsgroups(
                            subset='train',
                            categories=['sci.space','sci.med'],
                            remove = ('headers','footers','quotes')
                           )

documents = data.data[:500]

print(f"Loaded {len(documents)} documents")
print(documents[0][:200])


Loaded 500 documents
From: "Phil G. Fraering" <pgf@srl03.cacs.usl.edu>

Right, the Profiting Caste is blessed by God, and may 
 freely blare its presence in the evening twilight ..



In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

# This downloads a small model (~90MB), works on CPU
model = SentenceTransformer('all-MiniLM-L6-v2')

# Embed all 500 documents — takes ~1 minute on CPU
embeddings = model.encode(documents,show_progress_bar = True)
print(f"Embedding shape : {embeddings.shape}")  # Should be (500, 384)

# Save them so you don't re-compute
np.save('embeddings.npy', embeddings)


Batches: 100%|██████████| 16/16 [00:16<00:00,  1.00s/it]

Embedding shape : (500, 384)


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

def search(query,documents,embeddings,model,top_k=5):
    
    ##  Convert query to embedding
    query_embedding = model.encode([query])

    ##  Compare query to every document
    scores = cosine_similarity(query_embedding,embeddings)[0]

    ## get top results

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in top_indices:
        results.append({
            'document' : documents[idx][:300] , ## First 300 chars
            'score' : scores[idx],
            'index' : idx
        })
    return results

## Test it

results = search("heart disease treatment", documents, embeddings, model)
for r in results:
    print(f"Score: {r['score']:.3f}")
    print(r['document'])
    print("---")
    


Score: 0.394
It would be nice to think that individuals can somehow 'beat the system'
and like a space explorer, boldly go where no man has gone before and
return with a prize cure. Unfortunately, too often the prize is limited
and the efficacy of the 'cure' questionable when applied to all
sufferers.

This appl
---
Score: 0.368




Your doctor is right. It is best to do nothing, besides taking some pain
medication initially. Some patients don't like this and expect, or demand,
to have something done. In these cases some physicians will "tape" the 
patient (put a lot of heavy adhesive tape around the chest), or prescribe
an
---
Score: 0.351
-*----

Ming-zhou Liu's main problem is that he has an incompetent
physician -- himself.  This physician has diagnosed a problem,
even though he probably has never seen the diagnosed disease
before and has no idea of what kinds of problems can present
similar symptoms.  This physician now wants to t
---
Score: 0.339
Gordon Rubenfeld responds to Ron 

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Build TF-IDF index
tfidf = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf.fit_transform(documents)

def keyword_search(query,documents,tfidf,tfidf_matrix,top_k=5):
    query_vec = tfidf.transform([query])
    scores = cosine_similarity(query_vec,tfidf_matrix)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [{'document': documents[i][:300], 'score': scores[i]} for i in top_indices]


# Now compare both for the same query
query = "space shuttle mission"
print("=== SEMANTIC SEARCH ===")
for r in search(query, documents, embeddings, model):
    print(f"{r['score']:.3f}: {r['document'][:100]}")

print("\n=== KEYWORD SEARCH ===")
for r in keyword_search(query, documents, tfidf, tfidf_matrix):
    print(f"{r['score']:.3f}: {r['document'][:100]}")

=== SEMANTIC SEARCH ===
0.550: Archive-name: space/schedule
Last-modified: $Date: 93/04/01 14:39:23 $

SPACE SHUTTLE ANSWERS, LAUNC
0.478: For an essay, I am writing about the space shuttle and a need for a better
propulsion system.  Throu
0.475: I am looking for any information about the space program.
This includes NASA, the shuttles, history,
0.472: Archive-name: space/astronaut
Last-modified: $Date: 93/04/01 14:39:02 $

HOW TO BECOME AN ASTRONAUT

0.452: From the "JPL Universe"
April 23, 1993

VLBI project meets with international space agencies

=== KEYWORD SEARCH ===
0.307: 
Better idea for use of NASA Shuttle Astronauts and Crew is have them be found
lost in space after a
0.266: Archive-name: space/schedule
Last-modified: $Date: 93/04/01 14:39:23 $

SPACE SHUTTLE ANSWERS, LAUNC
0.255: For an essay, I am writing about the space shuttle and a need for a better
propulsion system.  Throu
0.206: There is an interesting opinion piece in the business section of today's
LA Times (Thursd

# Phase 1 : Scale the dataset to 5,000 docs

In [14]:
from sklearn.datasets import fetch_20newsgroups
from sentence_transformers import SentenceTransformer

import numpy as np
import faiss
import os

## Add more categories to get more documents
data = fetch_20newsgroups(
    subset='train',
    categories=['sci.space','sci.med','sci.electronics','sci.crypt'],
    remove=('headers','footers','quotes')
)

documents = [doc.strip() for doc in data.data if len(doc.strip()) > 100]
documents = documents[:5000]

print(f"Total Documents : {len(documents)}")


Total Documents : 2201


Step 2: Build and save the FAISS index

In [15]:
model = SentenceTransformer('all-MiniLM-L6-v2')

EMBEDDINGS_FILE = 'embeddings.npy'
INDEX_FILE = 'faiss.index'

# Only compute embeddings if not already saved

if os.path.exists(EMBEDDINGS_FILE):
    print("Loading saved embeddings ....")
    embeddings = np.load(EMBEDDINGS_FILE)
else:
    print("Computing embedding (one time only)...")
    embeddings = model.encode(documents,show_progress_bar=True,batch_size=32)
    embeddings = embeddings.astype('float32') # FAISS required float 32
    np.save(EMBEDDINGS_FILE,embeddings)
    print("embedding saved.")

print(f"Embeddings shpae : {embeddings.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2052.73it/s]


Computing embedding (one time only)...


Batches: 100%|██████████| 69/69 [01:25<00:00,  1.23s/it]

embedding saved.
Embeddings shpae : (2201, 384)


Step 3: Build the FAISS index

In [16]:
# Build FAISS index

if os.path.exists(INDEX_FILE):
    print("Loading saved FAISS index ...")
    index = faiss.read_index(INDEX_FILE)
else:
    print("Building FAISS index...")
    dimension = embeddings.shape[1] # 384

    # IndexFlatL2 = exact search using L2 distance
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    faiss.write_index(index,INDEX_FILE)
    print(f"FAISS index saved. Total vectors : {index.ntotal}")
    

Building FAISS index...
FAISS index saved. Total vectors : 2201


 Step 4: New search function using FAISS

In [20]:
def search(query,top_k=5):
    # Embed the query
    query_embedding = model.encode([query])
    query_embedding = query_embedding.astype('float32')

    #FAISS search - returns distances and indices
    distances, indices = index.search(query_embedding,top_k)
    
    results = []
    for dist , idx in zip(distances[0],indices[0]):
        results.append({
            'document': documents[idx][:300],
            'distance': dist,        # Lower = more similar in L2
            'index': idx
        })
    return results

# Test it
query = "symptoms of heart disease"
print(f"\nQuery: '{query}'\n")
for i, r in enumerate(search(query)):
    print(f"Result {i+1} | Distance: {r['distance']:.3f}")
    print(r['document'])
    print("---")



Query: 'symptoms of heart disease'

Result 1 | Distance: 1.142
The term arrhythmia is usually used to encompass a wide range of abnormal
heart rhythms (cardiac dysrhythmias).  Some of them are very serious
while others are completely benign.  Having "a few irregular beats"
on an EKG could be serious depending on what those beats were and
when they occurred, or 
---
Result 2 | Distance: 1.179
I don't know if anyone knows about this topic: electrical heart 
failure. One of my friends has had to go to the doctor because
he had chest pains. The Doc said it was Arythmia. So he had to
go to a new york hospital for a lot of money to get treated. His
doctors said that he could die from it, and 
---
Result 3 | Distance: 1.329
The funny thing is the personaly stories about reactions to MSG vary so
greatly. Some said that their heart beat speeded up with flush face. Some
claim their heart "skipped" beats once in a while. Some reacted with
headache, some stomach ache. Some had watery eyes or runn

## 🔍 Similarity vs Distance (Important Concept)

### 📌 Phase 0: Cosine Similarity
### ⚖️ Key Difference

| Method              | What it Measures | Similarity Rule        |
|--------------------|------------------|------------------------|
| Cosine Similarity  | Angle            | Higher = More similar  |
| L2 Distance        | Distance         | Lower = More similar   |

---

### 💡 Key Insight
- Both measure **how similar two vectors are**
- Just using **different perspectives**:
  - Cosine → direction  
  - L2 → distance  

👉 Same goal, different math

---

 ---
## 🚀 Why IndexFlatL2 Gets Slow at Scale

### 🔍 What is happening?
- FAISS compares query with every vector
- This is called **exact search**

### ⚠️ Problem
- 5,000 docs → 5,000 comparisons
- 1M docs → 1M comparisons

### 💡 Key Insight
- Accurate ✅
- Slow ❌

- Called **brute-force search**
- Very accurate ✅
- Very slow ❌
 ---

## 🚀 FAISS HNSW Index (Fast Approximate Search)

### 🧠 Big Picture
- You are using **HNSW (Hierarchical Navigable Small World)** for fast search
- Instead of comparing with all vectors (brute-force), it:
  - builds a **graph of neighbors**
  - searches by **navigating the graph**

👉 Faster than `IndexFlatL2`, slightly less accurate

---

In [22]:

if os.path.exists(INDEX_FILE):
    print("Loading saved HNSW index...")
    index = faiss.read_index(INDEX_FILE)
else : 
    print("Building HNSW index ...")
    dimension = embeddings.shape[1]

    # M = number of connections per node in the graph
    # Higher M = more accurate but more memory
    # 32 is the standard production default
    M = 32
    index = faiss.IndexHNSWFlat(dimension,M)

    # ef_construction = how carefully it builds the graph
    # Higher = better quality index, slower to build (one time cost)
    index.hnsw.efConstruction = 200
    
    index.add(embeddings)
    faiss.write_index(index,INDEX_FILE)
    print(f"HNSW index saved. Total vectors: {index.ntotal}")

# Set ef_search at query time
# Higher = more accurate search, slightly slower
# 50 is a good default
index.hnsw.efSearch = 50


Loading saved HNSW index...


In [24]:
import time

query = "heart attack"

# Time HNSW search
start = time.time()
for _ in range(100):  # Run 100 times to get stable measurement
    results = search(query, top_k=5)
hnsw_time = (time.time() - start) / 100

print(f"HNSW average search time: {hnsw_time*1000:.2f} ms")
print(f"\nTop result:")
print(results[0]['document'][:200])

HNSW average search time: 17.02 ms

Top result:
I don't know if anyone knows about this topic: electrical heart 
failure. One of my friends has had to go to the doctor because
he had chest pains. The Doc said it was Arythmia. So he had to
go to a n


In [26]:
# fetch_arxiv.py
import arxiv
import json
import time
import os